In [9]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [10]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [11]:
TARGET = 'PitNextLap'

print(df_train[TARGET].value_counts())
print(df_train_merged[TARGET].value_counts())

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    427273
1.0    113172
Name: count, dtype: int64


In [12]:
df_test=df_test.drop(['Driver'],axis=1)
df_train_merged=df_train_merged.drop(['Driver'],axis=1)
df_train=df_train.drop(['Driver'],axis=1)


In [13]:
# predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc').fit(
#     train_data=df_train,
#     ag_args_fit={"num_gpus": 2},
#     time_limit=3600*9,
#     presets='best_quality',
#     verbosity=3,
#     num_stack_levels=0
# )

In [14]:
import warnings
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*downcast.*"
)
_orig_fillna = pd.DataFrame.fillna
def _patched_fillna(self, *args, **kwargs):
    kwargs.pop("downcast", None)
    return _orig_fillna(self, *args, **kwargs)

In [ ]:
import os
import shutil
import pandas as pd
from autogluon.tabular import TabularPredictor

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "./ag_models3"
TIME_LIMIT = 30 * 60  # 30 minutes (1800 seconds) per model family

models = {
    "GBM": [
        {},  # Generates LightGBM_BAG_L1
        {"extra_trees": True, "ag_args": {"name_suffix": "Large"}},  # Generates LightGBMLarge_BAG_L1
    ],
    "XGB": {},  # Generates XGBoost_BAG_L1
    "CAT": {},  # Generates CatBoost_BAG_L1
}

# ============================================================
# DELETE OLD MODELS
# ============================================================

# if os.path.exists(MODEL_DIR):
#     print(f"Deleting existing model directory: {MODEL_DIR}")
#     shutil.rmtree(MODEL_DIR)

# os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Fresh model directory created: {MODEL_DIR}")

# ============================================================
# TRAIN ONE MODEL AT A TIME
# ============================================================

predictors = {}
results = []

for model_name, model_config in models.items():

    model_path = os.path.join(MODEL_DIR, model_name)

    print("\n" + "=" * 80)
    print(f"TRAINING: {model_name}")
    print(f"TIME LIMIT: 30 MINUTES")
    print(f"MODEL PATH: {model_path}")
    print("=" * 80)

    predictor = TabularPredictor(
        label=TARGET,
        eval_metric='roc_auc',
        path=model_path
    ).fit(
        train_data=df_train_merged,
        presets="best_quality",  # Enables bagging (_BAG_L1) and out-of-fold validation
        hyperparameters={model_name: model_config},  # Maps key to config dict properly
        ag_args_fit={
            'num_gpus': 1
        },
        time_limit=TIME_LIMIT,
        verbosity=3,
        num_stack_levels=0
    )

    predictors[model_name] = predictor

    # ========================================================
    # GET RESULT
    # ========================================================

    leaderboard = predictor.leaderboard(silent=True)
    best_row = leaderboard.iloc[0]

    results.append({
        'Model Family': model_name,
        'Best AutoGluon Model': best_row['model'],
        'Validation ROC-AUC': best_row['score_val'],
        'Fit Time (sec)': best_row['fit_time'],
        'Predict Time (sec)': best_row['pred_time_val']
    })

    print(f"\n{model_name} completed.")
    print(f"Validation ROC-AUC: {best_row['score_val']:.6f}")

# ============================================================
# FINAL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    'Validation ROC-AUC',
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display(results_df)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.13.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       2.99 GB / 15.06 GB (19.9%)
Disk Space Avail:   723.00 GB / 930.47 GB (77.7%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 

Fresh model directory created: ./ag_models3

TRAINING: GBM
TIME LIMIT: 30 MINUTES
MODEL PATH: ./ag_models3\GBM


	Available Memory:                    3161.44 MB
	Train Data (Original)  Memory Usage: 109.08 MB (3.5% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 1 features to boolean dtype as they only contain 2 unique values.
			Original Features (exact raw dtype, raw dtype):
				('float64', 'float') : 6 | ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', ...]
				('int64', 'int')     : 5 | ['Year', 'PitStop', 'LapNumber', 'Stint', 'Position']
				('object', 'object') : 2 | ['Compound', 'Race']
			Types of features in original data (raw dtype, special dtypes):
				('float', [])  : 6 | ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', ...]
				('int', [])    : 5 | ['Year', 'PitStop', 'LapNumber', 'Stint', 'Position']
				('object', []) : 2

[50]	valid_set's binary_logloss: 0.281916
[100]	valid_set's binary_logloss: 0.258851
[150]	valid_set's binary_logloss: 0.250295
[200]	valid_set's binary_logloss: 0.245593
[250]	valid_set's binary_logloss: 0.241892
[300]	valid_set's binary_logloss: 0.239033
[350]	valid_set's binary_logloss: 0.23689
[400]	valid_set's binary_logloss: 0.235024
[450]	valid_set's binary_logloss: 0.233251
[500]	valid_set's binary_logloss: 0.231941
[550]	valid_set's binary_logloss: 0.230582
[600]	valid_set's binary_logloss: 0.229416
[650]	valid_set's binary_logloss: 0.228356
[700]	valid_set's binary_logloss: 0.227139
[750]	valid_set's binary_logloss: 0.226295
[800]	valid_set's binary_logloss: 0.225553
[850]	valid_set's binary_logloss: 0.224755
[900]	valid_set's binary_logloss: 0.224193
[950]	valid_set's binary_logloss: 0.223656
[1000]	valid_set's binary_logloss: 0.22302
[1050]	valid_set's binary_logloss: 0.22256
[1100]	valid_set's binary_logloss: 0.222066
[1150]	valid_set's binary_logloss: 0.221527
[1200]	vali

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


...

	Fit constraints were validated upstream on the full training data, skipping...
	Training S1F8 with GPU, note that this may negatively impact model quality compared to CPU training.
	Catboost model hyperparameters: {'iterations': 10000, 'learning_rate': 0.05, 'allow_writing_files': False, 'eval_metric': 'Logloss', 'random_seed': 0, 'thread_count': 16, 'task_type': 'GPU'}


0:	learn: 0.6648191	test: 0.6647222	best: 0.6647222 (0)	total: 4.42ms	remaining: 4.42ms
1:	learn: 0.6391827	test: 0.6389874	best: 0.6389874 (1)	total: 9.23ms	remaining: 0us
bestTest = 0.6389874454
bestIteration = 1
0:	learn: 0.6398744	test: 0.6397168	best: 0.6397168 (0)	total: 15ms	remaining: 51.3s
20:	learn: 0.3402192	test: 0.3381433	best: 0.3381433 (20)	total: 311ms	remaining: 50.4s
40:	learn: 0.3092316	test: 0.3062750	best: 0.3062750 (40)	total: 614ms	remaining: 50.6s
60:	learn: 0.2970183	test: 0.2938685	best: 0.2938685 (60)	total: 919ms	remaining: 50.6s
80:	learn: 0.2888973	test: 0.2858265	best: 0.2858265 (80)	total: 1.23s	remaining: 50.5s
100:	learn: 0.2835498	test: 0.2804807	best: 0.2804807 (100)	total: 1.53s	remaining: 50.4s
120:	learn: 0.2794785	test: 0.2765137	best: 0.2765137 (120)	total: 1.83s	remaining: 50s
140:	learn: 0.2746772	test: 0.2718160	best: 0.2718160 (140)	total: 2.14s	remaining: 49.8s
160:	learn: 0.2714445	test: 0.2687823	best: 0.2687823 (160)	total: 2.45s	remaini

Saving c:\Darshak\Projects\Hackathon\ag_models3\CAT\models\CatBoost_BAG_L1\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models3\CAT\models\CatBoost_BAG_L1\model.pkl
	0.9435	 = Validation score   (roc_auc)
	181.82s	 = Training   runtime
	0.16s	 = Validation runtime
	411605.4	 = Inference  throughput (rows/s | 67556 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models3\CAT\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\models\CatBoost_BAG_L1\utils\oof.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'problem_types': ['binary', 'multiclass', 'regression', 'quantile', 'softclass'], 'valid_base': False, 'name_bag_suffix': '', 'model_type': <class 'autogluon.core.models.greedy_ensemble.greedy_weighted_ensemble_model.GreedyWeightedEnsembleModel'>, 'priority': 0}, 'ag_args_ensemble': {'save_bag_folds': True}}
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 1616.94s of remaining time.
	


CAT completed.
Validation ROC-AUC: 0.943546

FINAL MODEL COMPARISON


,Model Family,Best AutoGluon Model,Validation ROC-AUC,Fit Time (sec),Predict Time (sec)
0,GBM,WeightedEnsemble_L2,0.957781,410.961213,36.298102
1,XGB,XGBoost_BAG_L1,0.956154,255.022554,3.170212
2,CAT,CatBoost_BAG_L1,0.943546,181.816644,0.164128


In [18]:
import os
import pandas as pd
from autogluon.tabular import TabularPredictor

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "./ag_models3"

all_leaderboards = []

# ============================================================
# SCAN & LOAD ALL TRAINED PREDICTORS
# ============================================================

if os.path.exists(MODEL_DIR):
    # Check both subdirectories and root directory for saved predictors
    folder_candidates = [MODEL_DIR] + [
        os.path.join(MODEL_DIR, d) 
        for d in os.listdir(MODEL_DIR) 
        if os.path.isdir(os.path.join(MODEL_DIR, d))
    ]

    for folder_path in folder_candidates:
        # Check if directory contains a valid predictor file
        if os.path.exists(os.path.join(folder_path, "predictor.pkl")):
            try:
                folder_name = os.path.basename(folder_path)
                print(f"Loading predictor from: {folder_path}")
                
                # Load predictor from disk
                predictor = TabularPredictor.load(folder_path)
                
                # Fetch leaderboard
                lb = predictor.leaderboard(silent=True)
                lb.insert(0, "folder_name", folder_name)
                
                all_leaderboards.append(lb)
            except Exception as e:
                print(f"Failed to load predictor from {folder_path}: {e}")

# ============================================================
# MERGE AND PRESENT COMBINED LEADERBOARD
# ============================================================

if all_leaderboards:
    combined_leaderboard = pd.concat(all_leaderboards, ignore_index=True)

    # Sort all trained models by validation ROC-AUC score descending
    combined_leaderboard = combined_leaderboard.sort_values(
        by="score_val", ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 80)
    print("COMBINED LEADERBOARD OF ALL LOADED MODELS")
    print("=" * 80)

    display(combined_leaderboard)
else:
    print(f"No valid AutoGluon predictors (`predictor.pkl`) found under '{MODEL_DIR}'.")

Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\models\trainer.pkl


Loading predictor from: ./ag_models3\CAT
Loading predictor from: ./ag_models3\GBM


Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\trainer.pkl


Loading predictor from: ./ag_models3\XGB

COMBINED LEADERBOARD OF ALL LOADED MODELS


,folder_name,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,GBM,WeightedEnsemble_L2,0.957781,roc_auc,36.298102,410.961213,0.051081,2.553276,2,True,3
1,GBM,LightGBM_BAG_L1,0.957254,roc_auc,12.108692,179.215388,12.108692,179.215388,1,True,1
2,XGB,XGBoost_BAG_L1,0.956154,roc_auc,3.170212,255.022554,3.170212,255.022554,1,True,1
3,XGB,WeightedEnsemble_L2,0.956154,roc_auc,3.220778,255.079135,0.050566,0.056581,2,True,2
4,GBM,LightGBMLarge_BAG_L1,0.955598,roc_auc,24.138329,229.192549,24.138329,229.192549,1,True,2
5,CAT,WeightedEnsemble_L2,0.943546,roc_auc,0.214443,181.873038,0.050315,0.056394,2,True,2
6,CAT,CatBoost_BAG_L1,0.943546,roc_auc,0.164128,181.816644,0.164128,181.816644,1,True,1


In [32]:
predictor = TabularPredictor.load("ag_models3/GBM")
predictor.info()

Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F3\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F4\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F5\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F6\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\S1F7\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\L

{'path': 'c:\\Darshak\\Projects\\Hackathon\\ag_models3\\GBM',
 'label': 'PitNextLap',
 'random_state': 0,
 'version': '1.6.1',
 'features': ['Compound',
  'Race',
  'Year',
  'PitStop',
  'LapNumber',
  'Stint',
  'TyreLife',
  'Position',
  'LapTime (s)',
  'LapTime_Delta',
  'Cumulative_Degradation',
  'RaceProgress',
  'Position_Change'],
 'feature_metadata_in': <autogluon.common.features.feature_metadata.FeatureMetadata at 0x27b80ebb7d0>,
 'time_fit_preprocessing': 0.7483129501342773,
 'time_fit_training': 451.31227707862854,
 'time_fit_total': 452.0605900287628,
 'time_limit': 1800,
 'time_train_start': 1786365301.0065231,
 'num_rows_train': 540445,
 'num_cols_train': 13,
 'num_rows_val': None,
 'num_rows_test': None,
 'num_classes': 2,
 'problem_type': 'binary',
 'eval_metric': 'roc_auc',
 'best_model': 'WeightedEnsemble_L2',
 'best_model_score_val': np.float64(0.9577806665332587),
 'best_model_stack_level': 2,
 'num_models_trained': 3,
 'num_bag_folds': 8,
 'max_stack_level': 2,

In [33]:
df=predictor.predict_proba(df_test)
df.head()

Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\WeightedEnsemble_L2\model.pkl


,0,1
0,0.995913,0.004087
1,0.994767,0.005233
2,0.996719,0.003281
3,0.749973,0.250027
4,0.079022,0.920978


In [34]:
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [35]:
df_sample_out['PitNextLap']=df[1]

In [36]:
df_sample_out.head()

,id,PitNextLap
0,439140,0.004087
1,439141,0.005233
2,439142,0.003281
3,439143,0.250027
4,439144,0.920978


In [37]:
df_sample_out.to_csv("My_output/Without_driver_feature_submission.csv")

In [38]:
### My sampling testing
from multi_sampling_test_predictor import process_and_evaluate_all_csvs
data_dict = process_and_evaluate_all_csvs(predictor,folder_path="Sampling_data_to_test")

Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBMLarge_BAG_L1\model.pkl



----------------------------------------
 Processing: sample_part_1_10000k.csv
----------------------------------------
ground_truth
0    8000
1    2000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBMLarge_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_1_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9439
Mismatches                         : 561
Accuracy (%)                       : 94.39
True Positives (Actual 1, Pred 1)  : 1705
True Negatives (Actual 0, Pred 0)  : 7734
False Positives (Actual 0, Pred 1) : 266
False Negatives (Actual 1, Pred 0) : 295

----------------------------------------
 Processing: sample_part_2_10000k.csv
----------------------------------------
ground_truth
0    8500
1    1500
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBMLarge_BAG_L1\model.pkl


...

--- Per-File Analysis [sample_part_3_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9401
Mismatches                         : 599
Accuracy (%)                       : 94.01
True Positives (Actual 1, Pred 1)  : 2152
True Negatives (Actual 0, Pred 0)  : 7249
False Positives (Actual 0, Pred 1) : 251
False Negatives (Actual 1, Pred 0) : 348

----------------------------------------
 Processing: sample_part_4_10000k.csv
----------------------------------------
ground_truth
0    7000
1    3000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBMLarge_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_4_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9406
Mismatches                         : 594
Accuracy (%)                       : 94.06
True Positives (Actual 1, Pred 1)  : 2616
True Negatives (Actual 0, Pred 0)  : 6790
False Positives (Actual 0, Pred 1) : 210
False Negatives (Actual 1, Pred 0) : 384

----------------------------------------
 Processing: sample_part_5_10000k.csv
----------------------------------------
ground_truth
0    9000
1    1000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\WeightedEnsemble_L2\model.pkl


--- Per-File Analysis [sample_part_5_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9560
Mismatches                         : 440
Accuracy (%)                       : 95.6
True Positives (Actual 1, Pred 1)  : 843
True Negatives (Actual 0, Pred 0)  : 8717
False Positives (Actual 0, Pred 1) : 283
False Negatives (Actual 1, Pred 0) : 157

      OVERALL CUMULATIVE ANALYSIS REPORT        
      PREDICTION ANALYSIS REPORT        
Total Records                      : 50000
Correct Matches                    : 47301
Mismatches                         : 2699
Accuracy (%)                       : 94.6
True Positives (Actual 1, Pred 1)  : 8607
True Negatives (Actual 0, Pred 0)  : 38694
False Positives (Actual 0, Pred 1) : 1306
False Negatives (Actual 1, Pred 0) : 1393
